In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="PR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="PR")


model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.33084768056869507
epoch:  1, loss: 0.1562277227640152
epoch:  2, loss: 0.06928230822086334
epoch:  3, loss: 0.040864381939172745
epoch:  4, loss: 0.0331852063536644
epoch:  5, loss: 0.031252630054950714
epoch:  6, loss: 0.0307668037712574
epoch:  7, loss: 0.03062695823609829
epoch:  8, loss: 0.030571769922971725
epoch:  9, loss: 0.030384086072444916
epoch:  10, loss: 0.030384086072444916
epoch:  11, loss: 0.03027971461415291
epoch:  12, loss: 0.0302207600325346
epoch:  13, loss: 0.03008769452571869
epoch:  14, loss: 0.03008769452571869
epoch:  15, loss: 0.02995201013982296
epoch:  16, loss: 0.029878143221139908
epoch:  17, loss: 0.02975715510547161
epoch:  18, loss: 0.02975715510547161
epoch:  19, loss: 0.029575923457741737
epoch:  20, loss: 0.029482679441571236
epoch:  21, loss: 0.02939695306122303
epoch:  22, loss: 0.02939695306122303
epoch:  23, loss: 0.029161347076296806
epoch:  24, loss: 0.029043495655059814
epoch:  25, loss: 0.029010841622948647
epoch:  26, los

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7365823712302875
Test metrics:  R2 = 0.704475172619162


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="FR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="FR")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3941551148891449
epoch:  1, loss: 0.2337033748626709
epoch:  2, loss: 0.03263302892446518
epoch:  3, loss: 0.032418157905340195
epoch:  4, loss: 0.0321650393307209
epoch:  5, loss: 0.03216099739074707
epoch:  6, loss: 0.032156046479940414
epoch:  7, loss: 0.03215603157877922
epoch:  8, loss: 0.03215597942471504
epoch:  9, loss: 0.03215499222278595
epoch:  10, loss: 0.032154615968465805
epoch:  11, loss: 0.032154615968465805
epoch:  12, loss: 0.032154615968465805
epoch:  13, loss: 0.032154615968465805
epoch:  14, loss: 0.032154615968465805
epoch:  15, loss: 0.032154615968465805
epoch:  16, loss: 0.032154615968465805
epoch:  17, loss: 0.032154615968465805
epoch:  18, loss: 0.032154615968465805
epoch:  19, loss: 0.032154615968465805
epoch:  20, loss: 0.032154615968465805
epoch:  21, loss: 0.032154615968465805
epoch:  22, loss: 0.032154615968465805
epoch:  23, loss: 0.032154615968465805
epoch:  24, loss: 0.032154615968465805
epoch:  25, loss: 0.032154615968465805
epoch: 

In [13]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -9100.555488589089
Test metrics:  R2 = -10119.538235383361


In [15]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    lr=10,
    c1=1e-4,
    tau=0.1,
    line_search_method="bisect",
    line_search_cond="wolfe",
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2313476949930191
epoch:  1, loss: 0.04176843538880348
epoch:  2, loss: 0.03452715277671814
epoch:  3, loss: 0.032785940915346146
epoch:  4, loss: 0.03247295692563057
epoch:  5, loss: 0.03238023817539215
epoch:  6, loss: 0.032298773527145386
epoch:  7, loss: 0.032298773527145386
epoch:  8, loss: 0.03225710615515709
epoch:  9, loss: 0.032234713435173035
epoch:  10, loss: 0.032234713435173035
epoch:  11, loss: 0.032180942595005035
epoch:  12, loss: 0.03193172067403793
epoch:  13, loss: 0.03193172067403793
epoch:  14, loss: 0.031918324530124664
epoch:  15, loss: 0.03166527673602104
epoch:  16, loss: 0.03166527673602104
epoch:  17, loss: 0.031647372990846634
epoch:  18, loss: 0.031627357006073
epoch:  19, loss: 0.031627357006073
epoch:  20, loss: 0.03157845512032509
epoch:  21, loss: 0.03123774379491806
epoch:  22, loss: 0.03123774379491806
epoch:  23, loss: 0.03121265396475792
epoch:  24, loss: 0.030766073614358902
epoch:  25, loss: 0.030989941209554672
epoch:  26, loss:

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -659.6809915480883
Test metrics:  R2 = -657.3517873681222
